# **JEV FROM SCRATCH**


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import pandas as pd
import numpy as np
import json

import warnings
import math


In [2]:
warnings.filterwarnings("ignore", category = FutureWarning)
torch.set_float32_matmul_precision("high")


### DEVICE 

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
print(f"Device in using: {device}")


Device in using: cuda


## PARAMETER SPECIFICATION

In [4]:
vocab_input = 14000
vocab_output = vocab_input

d_model = 256
d_ff = d_model * 4 
num_heads = 4
num_layers = 6

max_output_len = 256


dropout = 0.1
threshold = 0.7
max_iterations = 4


### **POSITIONAL ENCODING**

In [5]:
def pos_encoding(seq_length, d_model):
    
    pe = torch.zeros(seq_length, d_model)
    pos = torch.arange(0, seq_length, dtype = torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000)/d_model))
    
    pe[:, 0::2] = torch.sin(pos * div_term)
    pe[:, 1::2] = torch.cos(pos * div_term)
    
    return pe


### **ENCODER SINGLE LAYER**

In [6]:
class encoder_single_layer(nn.Module):
    
    def __init__(self, d_model = d_model, d_ff = d_ff, dropout = dropout, vocab_input = vocab_input, num_heads = num_heads):
        super(encoder_single_layer, self).__init__()
        
        # LayerNorm
        
        self.norm = nn.LayerNorm(d_model)
        
        # MHA
        
        self.mha = nn.MultiheadAttention(d_model, num_heads = num_heads, batch_first = True)
        
        # norm
        
        self.norm2 = nn.LayerNorm(d_model)
        
        # ffn
        
        self.ffn = nn.Sequential(
            
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            
            nn.Linear(d_ff, d_model)
        )
        
        # norm
        
        self.norm3 = nn.LayerNorm(d_model)
        
        # dropout
        
        self.drop = nn.Dropout(dropout)
        
        # weight init
        
        self.apply(self.init_weight)
        
    def init_weight(self, m):
        
        if isinstance(m, nn.Linear):
            
            nn.init.xavier_normal_(m.weight, 1)
            
            if m.bias is not None:
                
                nn.init.zeros_(m.bias)
                
        if isinstance(m, nn.MultiheadAttention):
            
            nn.init.xavier_normal_(m.in_proj_weight, 1)
            
            if m.in_proj_bias is not None:
                
                nn.init.zeros_(m.in_proj_bias)
                
            nn.init.xavier_normal_(m.out_proj.weight, 1)
            
            
            if m.out_proj.bias is not None:
                
                nn.init.zeros_(m.out_proj.bias)
        
    def forward(self, x, mask = None):
        
        # norm
        
        norm = self.norm(x)
        
        # mha
        
        attn, weight = self.mha(norm, norm, norm, attn_mask = mask)
        
        # res
        
        x = self.drop(attn + x)
        
        # norm
        
        norm2 = self.norm2(x)
        
        # ffn
        
        ffn = self.ffn(norm2)
        
        # res
        
        x = self.drop(x + ffn)
        
        # normalized return
        
        x = self.norm3(x)
        
        return x
    
    def create_mask(seq_length, device):
        
        mask = torch.triu(torch.ones(seq_length, seq_length, dtype = torch.bool), diagonal = 1)
        
        return mask.to(device)


### **ENCODER STACK**

In [7]:
class encoder_stack(nn.Module):
    
    def __init__(self, num_layers = num_layers):
        super(encoder_stack, self).__init__()
    
        
        self.layers = nn.ModuleList([
            
            encoder_single_layer().to(device) for _ in range(num_layers)
            
        ])
        
        
    def forward(self, x):
        
        seq_length = x.size(1)
        
        mask = encoder_single_layer.create_mask(seq_length = seq_length, device = x.device)
        
        for layer in self.layers:
            
            x = layer(x, mask)
            
        return x


## **ENCODER**

In [8]:
class encoder(nn.Module):
    
    def __init__(self, vocab_input = vocab_input, d_model = d_model):
        super(encoder, self).__init__()
        
        self.d_model = d_model
        
        self.embed = nn.Embedding(vocab_input, d_model)
        
        self.encoder_stack = encoder_stack()

        
    def forward(self, x):
        
        seq = x.size(1)
        
        pos = pos_encoding(seq, self.d_model).to(device)
        
        embed = self.embed(x) * math.sqrt(self.d_model)
        
        x = embed + pos
        
        x = self.encoder_stack(x)
        
        return x
        

### **INIT**

In [9]:
ENCODER = encoder().to(device)
print(ENCODER)
print("-" * 80)

params = sum(p.numel() for p in ENCODER.parameters())
print(f"Number of parameters in the Encoder: {params:,}") 


encoder(
  (embed): Embedding(14000, 256)
  (encoder_stack): encoder_stack(
    (layers): ModuleList(
      (0-5): 6 x encoder_single_layer(
        (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (mha): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (ffn): Sequential(
          (0): Linear(in_features=256, out_features=1024, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=1024, out_features=256, bias=True)
        )
        (norm3): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (drop): Dropout(p=0.1, inplace=False)
      )
    )
  )
)
--------------------------------------------------------------------------------
Number of parameters in the Encoder: 8,325,632


### **SAFE TEST**

In [10]:
dummy_input = torch.randint(0, vocab_input, (1, 1400), device=device)

x = ENCODER(dummy_input)

print(x.shape)


torch.Size([1, 1400, 256])


### **PRE OUTPUT SLOTS**

In [11]:
class output_slots(nn.Module):
    
    def __init__(self, max_output_len = max_output_len, d_model = d_model):
        super(output_slots, self).__init__()
        
        self.slots = nn.Parameter(torch.randn(1, max_output_len, d_model))
        
    def forward(self, batch_size):
        
        x = self.slots.expand(batch_size, -1, -1)
        
        return x


In [12]:
OUTPUT_SLOTS = output_slots().to(device)

x = OUTPUT_SLOTS(batch_size=2)

print(x.shape)


torch.Size([2, 256, 256])


### **CROSS ATTN**

OUTPUT SLOTS ATTENDING THE ENCODER OUTPUT

In [13]:
class cross_attention(nn.Module):
    
    def __init__(self, d_model = d_model, dropout = dropout, num_heads = num_heads, d_ff = d_ff):
        super(cross_attention, self).__init__()
        
        self.norm = nn.LayerNorm(d_model)
        
        self.mha = nn.MultiheadAttention(d_model, num_heads, dropout, batch_first = True)
        
        self.norm2 = nn.LayerNorm(d_model)
        
        self.ffn = nn.Sequential(
            
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
            
        )
        
        self.drop = nn.Dropout(dropout)
        
        self.apply(self.__init_weight__)
        
    def __init_weight__(self, m):
        
        if isinstance(m, nn.Linear):
            
            nn.init.xavier_normal_(m.weight)
            
            if m.bias is not None:
                
                nn.init.zeros_(m.bias)
        
        if isinstance(m, nn.MultiheadAttention):
            
            nn.init.xavier_normal_(m.in_proj_weight)
            
            if m.in_proj_bias is not None:
                
                nn.init.zeros_(m.in_proj_bias)
                
            nn.init.xavier_normal_(m.out_proj.weight)
            
            if m.out_proj.bias is not None:
                
                nn.init.zeros_(m.out_proj.bias)
    
    def forward(self, slots, Encoder_out):
        
        norm = self.norm(slots)
        
        attn, weight = self.mha(norm, Encoder_out, Encoder_out)
        
        slots = slots + self.drop(attn)
        
        norm2 = self.norm2(slots)
        
        ffn = self.ffn(norm2)
        
        slots = slots + self.drop(ffn)
        
        return slots
                

In [14]:
CROSS_ATTN = cross_attention().to(device)


### **OUTPUT HEAD PROBAB**

In [15]:
class output_head(nn.Module):
    
    def __init__(self, d_model = d_model, vocab_output = vocab_output):
        super(output_head, self).__init__()
        
        self.norm = nn.LayerNorm(d_model)
        
        self.out = nn.Linear(d_model, vocab_output)
        
        self.apply(self.__init_weight__)
        
    def __init_weight__(self, m):
        
        if isinstance(m, nn.Linear):
            
            nn.init.xavier_normal_(m.weight)
            
            nn.init.zeros_(m.bias)
            
    def forward(self, x):
        
        norm = self.norm(x)
        
        logits = self.out(norm)
        
        return logits


In [16]:
OUTPUT_HEAD = output_head().to(device)

x = torch.randn(2, 16, d_model, device=device)

logits = OUTPUT_HEAD(x)

print(logits.shape)


torch.Size([2, 16, 14000])


## **TOKEN CONFIDENCE**

In [17]:
class token_confidence(nn.Module):
    
    def __init__(self):
        super(token_confidence, self).__init__()
        
    def forward(self, logits):
        
        probs = torch.softmax(logits, dim = -1)
        
        confidence, token = torch.max(probs, dim = -1)
        
        return confidence, token
        

In [18]:
CONFIDENCE = token_confidence().to(device)

logits = torch.randn(
    2,
    16,
    vocab_output,
    device=device
)

tokens, confidence = CONFIDENCE(logits)

print("Logits:", logits.shape)
print("Tokens:", tokens.shape)
print("Confidence:", confidence.shape)


Logits: torch.Size([2, 16, 14000])
Tokens: torch.Size([2, 16])
Confidence: torch.Size([2, 16])


### **REFINEMENT MASK**

In [19]:
class refinement_mask(nn.Module):
    
    def __init__(self, threshold = threshold):
        super(refinement_mask, self).__init__()
        
        self.threshold = threshold
        
    def forward(self, confidence):
        
        mask = confidence < self.threshold
        
        return mask


In [20]:
REFINEMENT_MASK = refinement_mask(
    threshold=0.7
).to(device)

mask = REFINEMENT_MASK(confidence)

print(mask.shape)
print(mask)


torch.Size([2, 16])
tensor([[False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False]], device='cuda:0')


## **SLOT REFINER**

In [21]:
class slot_refiner(nn.Module):
    
    def __init__(self, d_model = d_model):
        super(slot_refiner, self).__init__()
        
        self.max_token = nn.Parameter(torch.rand(1, 1, d_model))
        
    def forward(self, slots, mask):
        
        mask_token = self.max_token.expand(slots.size(0), slots.size(1), -1)
        
        refined_slots = torch.where(mask.unsqueeze(1), mask_token, slots)
        
        return refined_slots


In [22]:
SLOT_REFINER = slot_refiner().to(device)


## **REFINEMENT LOOP**

In [23]:
class refinement_loop(nn.Module):
    
    def __init__(self, max_iterations = max_iterations):
        super(refinement_loop, self).__init__()
        
        self.max_iterations = max_iterations
        
    def forward(self, slots, encoder_out, refinement_mask, slot_refiner):
        
        for _ in range(self.max_iterations):
            
            slots = CROSS_ATTN(slots, encoder_out)
            
            logits = OUTPUT_HEAD(slots)
            
            token, confidence = CONFIDENCE(logits)
            
            mask = refinement_mask(confidence)
            
            slots = slot_refiner(slots, mask)
            
            return token, confidence, logits


## **TERMINATION**

In [24]:
class termination(nn.Module):
    
    def __init__(self, threshold = threshold):
        super(termination, self).__init__()
        
        self.threshold = threshold
        
    def forward(self, confidence):
        
        done = (confidence >=  self.threshold).all(dim = 1)
        
        return done


In [25]:
TERMINATION = termination().to(device)


In [26]:
batch_size = 2
input_seq = 32
output_seq = 16

dummy_input = torch.randint(
    0,
    vocab_input,
    (batch_size, input_seq),
    device=device
)

# Encoder
encoder_output = ENCODER(dummy_input)

print("Encoder:", encoder_output.shape)

# Output slots
slots = OUTPUT_SLOTS(batch_size)

print("Slots:", slots.shape)

# One complete refinement pass

slots = CROSS_ATTN(
    slots,
    encoder_output
)

print("After Cross Attention:", slots.shape)

# Vocabulary prediction
logits = OUTPUT_HEAD(slots)

print("Logits:", logits.shape)

# Confidence
tokens, confidence = CONFIDENCE(logits)

print("Tokens:", tokens.shape)
print("Confidence:", confidence.shape)

# Refinement mask
mask = REFINEMENT_MASK(confidence)

print("Refinement Mask:", mask.shape)

# Refine slots
slots = SLOT_REFINER(
    slots,
    mask
)

print("Refined Slots:", slots.shape)

# Termination
done = TERMINATION(confidence)

print("Done:", done.shape)
print("Done:", done)


Encoder: torch.Size([2, 32, 256])


Slots: torch.Size([2, 256, 256])
After Cross Attention: torch.Size([2, 256, 256])
Logits: torch.Size([2, 256, 14000])
Tokens: torch.Size([2, 256])
Confidence: torch.Size([2, 256])
Refinement Mask: torch.Size([2, 256])
Refined Slots: torch.Size([2, 256, 256])
Done: torch.Size([2])
Done: tensor([True, True], device='cuda:0')


## **JEV WRAPPER**

In [27]:
class jev(nn.Module):
    
    def __init__(self, ENCODER = ENCODER, OUTPUT_SLOTS = OUTPUT_SLOTS, CROSS_ATTN = CROSS_ATTN, CONFIDENCE = CONFIDENCE, OUTPUT_HEAD = OUTPUT_HEAD, SLOT_REFINER = SLOT_REFINER, TERMINATION = TERMINATION, MAX_ITERATION = max_iterations, REFINEMENT_MASK = REFINEMENT_MASK):
        super(jev, self).__init__()
        
        self.encoder = ENCODER
        self.output_slots = OUTPUT_SLOTS
        self.output_head = OUTPUT_HEAD
        self.cross_attn = CROSS_ATTN
        self.confidence = CONFIDENCE
        self.slot_refinder = SLOT_REFINER
        self.termination = TERMINATION
        self.refinement_mask = REFINEMENT_MASK
        self.max_iteration = max_iterations
        
    def forward(self, x):
        
        encoder_out = self.encoder(x)
        
        slots = self.output_slots(batch_size = x.size(0))
        
        finished = torch.zeros(x.size(0), dtype = torch.bool, device = x.device)
        
        for step in range(self.max_iteration):

            # Cross attention + FFN
            
            slots = self.cross_attn(
                slots,
                encoder_output
            )

            # Predict vocabulary
            
            logits = self.output_head(slots)

            # Token + confidence
            
            tokens, confidence = self.confidence(
                logits
            )

            # Find uncertain positions
            
            mask = self.refinement_mask(
                confidence
            )

            # Check termination
            
            finished = self.termination(
                confidence
            )

            # Stop if every sample is finished
            
            if finished.all():
                break

            # Refine uncertain slots
            
            slots = self.slot_refiner(
                slots,
                mask
            )
            
        return tokens, logits, confidence


In [28]:
JEV = jev().to(device)

dummy_input = torch.randint(
    0,
    vocab_input,
    (2, 32),
    device=device
)

tokens, logits, confidence = JEV(
    dummy_input
)

print("Tokens:", tokens.shape)
print("Logits:", logits.shape)
print("Confidence:", confidence.shape)


Tokens: torch.Size([2, 256])
Logits: torch.Size([2, 256, 14000])
Confidence: torch.Size([2, 256])


In [ ]:
print(JEV)
print("-" * 70)
params = sum(p.numel() for p in JEV.parameters())
print(f"Params in jev: {params:,}")


jev(
  (encoder): encoder(
    (embed): Embedding(14000, 256)
    (encoder_stack): encoder_stack(
      (layers): ModuleList(
        (0-5): 6 x encoder_single_layer(
          (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (mha): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (ffn): Sequential(
            (0): Linear(in_features=256, out_features=1024, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=1024, out_features=256, bias=True)
          )
          (norm3): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (drop): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (output_slots): output_slots()
  (output_head): output_head(
    (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (out): Linear(in_features=256,

# **DATA PROCESSING AND TRAINING**


### **ARCHITECTURE TRAINING FIX**

Small fixes needed before training the wrapper.


In [ ]:
class slot_refiner(nn.Module):
    
    def __init__(self, d_model = d_model):
        super(slot_refiner, self).__init__()
        
        self.mask_token = nn.Parameter(torch.rand(1, 1, d_model))
        
    def forward(self, slots, mask):
        
        mask_token = self.mask_token.expand(slots.size(0), slots.size(1), -1)
        
        refined_slots = torch.where(mask.unsqueeze(-1), mask_token, slots)
        
        return refined_slots


class jev(nn.Module):
    
    def __init__(self, ENCODER, OUTPUT_SLOTS, CROSS_ATTN, CONFIDENCE, OUTPUT_HEAD, SLOT_REFINER, TERMINATION, REFINEMENT_MASK, max_iteration = max_iterations):
        super(jev, self).__init__()
        
        self.encoder = ENCODER
        self.output_slots = OUTPUT_SLOTS
        self.output_head = OUTPUT_HEAD
        self.cross_attn = CROSS_ATTN
        self.confidence = CONFIDENCE
        self.slot_refiner = SLOT_REFINER
        self.termination = TERMINATION
        self.refinement_mask = REFINEMENT_MASK
        self.max_iteration = max_iteration
        
    def forward(self, x):
        
        encoder_out = self.encoder(x)
        slots = self.output_slots(batch_size = x.size(0))
        
        for step in range(self.max_iteration):
            
            slots = self.cross_attn(slots, encoder_out)
            logits = self.output_head(slots)
            confidence, tokens = self.confidence(logits)
            
            if self.termination(confidence).all():
                break
                
            mask = self.refinement_mask(confidence)
            slots = self.slot_refiner(slots, mask)
            
        return tokens, logits, confidence


SLOT_REFINER = slot_refiner().to(device)
JEV = jev(
    ENCODER,
    OUTPUT_SLOTS,
    CROSS_ATTN,
    CONFIDENCE,
    OUTPUT_HEAD,
    SLOT_REFINER,
    TERMINATION,
    REFINEMENT_MASK
).to(device)


### **DATA PATH AND PARAMETER**


In [ ]:
from pathlib import Path
from collections import Counter
from torch.utils.data import Dataset, DataLoader
import re
import random


project_path = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'bookcorpusopen' / 'data').exists())
data_path = project_path / 'bookcorpusopen' / 'data'
parquet_files = sorted(data_path.glob('*.parquet'))

batch_size = 8
input_length = 256
output_length = max_output_len
learning_rate = 3e-4
epochs = 3
max_books = 20000

print(f'Parquet files: {len(parquet_files)}')
print(f'Data path: {data_path}')


### **LOAD BOOKCORPUS DATA**


In [ ]:
def load_bookcorpus(parquet_files, max_books = None):
    
    books = []
    
    for file in parquet_files:
        
        frame = pd.read_parquet(file, columns = ['text'])
        texts = frame['text'].dropna().astype(str).tolist()
        books.extend([text for text in texts if text.strip()])
        
        if max_books is not None and len(books) >= max_books:
            books = books[:max_books]
            break
            
    return books


books = load_bookcorpus(parquet_files, max_books = max_books)
print(f'Books loaded: {len(books):,}')
print(books[0][:300])


### **TOKENIZER AND VOCABULARY**


In [ ]:
def tokenize(text):
    
    text = text.lower()
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?|[0-9]+|[^\s\w]", text)
    
    return tokens


PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'

word_counter = Counter()

for book in books:
    word_counter.update(tokenize(book))
    
vocab_words = [word for word, count in word_counter.most_common(vocab_input - 2)]
id_to_token = [PAD_TOKEN, UNK_TOKEN] + vocab_words
token_to_id = {token: idx for idx, token in enumerate(id_to_token)}

pad_id = token_to_id[PAD_TOKEN]
unk_id = token_to_id[UNK_TOKEN]

print(f'Vocabulary size: {len(token_to_id):,}')
print(f'Pad id: {pad_id} | Unknown id: {unk_id}')


### **TRAINING DATASET**

The first 256 tokens are encoder input and the next 256 tokens are the target output slots.


In [ ]:
class bookcorpus_dataset(Dataset):
    
    def __init__(self, books, token_to_id, input_length, output_length):
        
        self.samples = []
        self.input_length = input_length
        self.output_length = output_length
        self.pad_id = token_to_id[PAD_TOKEN]
        
        sequence_length = input_length + output_length
        
        for book in books:
            
            ids = [token_to_id.get(token, unk_id) for token in tokenize(book)]
            
            for start in range(0, len(ids) - 1, sequence_length):
                
                chunk = ids[start:start + sequence_length]
                
                if len(chunk) <= input_length:
                    continue
                    
                chunk = chunk + [self.pad_id] * (sequence_length - len(chunk))
                self.samples.append(chunk[:sequence_length])
                
    def __len__(self):
        
        return len(self.samples)
    
    def __getitem__(self, index):
        
        sample = torch.tensor(self.samples[index], dtype = torch.long)
        
        input_ids = sample[:self.input_length]
        target_ids = sample[self.input_length:]
        
        return input_ids, target_ids


dataset = bookcorpus_dataset(
    books,
    token_to_id,
    input_length,
    output_length
)

train_loader = DataLoader(
    dataset,
    batch_size = batch_size,
    shuffle = True,
    num_workers = 0,
    pin_memory = torch.cuda.is_available()
)

print(f'Training samples: {len(dataset):,}')


### **OPTIMIZER AND LOSS**


In [ ]:
optimizer = optim.AdamW(
    JEV.parameters(),
    lr = learning_rate,
    weight_decay = 0.01
)

criterion = nn.CrossEntropyLoss(
    ignore_index = pad_id
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max = epochs
)


### **TRAINING LOOP**


In [ ]:
JEV.train()

for epoch in range(epochs):
    
    total_loss = 0
    
    for step, (input_ids, target_ids) in enumerate(train_loader):
        
        input_ids = input_ids.to(device, non_blocking = True)
        target_ids = target_ids.to(device, non_blocking = True)
        
        optimizer.zero_grad(set_to_none = True)
        
        tokens, logits, confidence = JEV(input_ids)
        
        loss = criterion(
            logits.reshape(-1, vocab_output),
            target_ids.reshape(-1)
        )
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(JEV.parameters(), max_norm = 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        
        if (step + 1) % 100 == 0:
            print(f'Epoch: {epoch + 1}/{epochs} | Step: {step + 1}/{len(train_loader)} | Loss: {total_loss / (step + 1):.4f}')
            
    scheduler.step()
    epoch_loss = total_loss / len(train_loader)
    
    print(f'Finished epoch {epoch + 1}/{epochs} | Average loss: {epoch_loss:.4f}')


### **SAVE TRAINED MODEL**


In [ ]:
checkpoint = {
    'model_state_dict': JEV.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'token_to_id': token_to_id,
    'id_to_token': id_to_token,
    'input_length': input_length,
    'output_length': output_length
}

torch.save(checkpoint, 'jev_bookcorpus_checkpoint.pt')
print('Training checkpoint saved.')
